# 6. Targets from astronomical catalogues

LUMOS reads astronomical catalogues (SIMBAD, Gaia, VizieR, the NASA Exoplanet Archive, JPL Horizons,
CALSPEC, SkyView...) and turns their answers into **typed records** with units, uncertainties and the
provenance of every value, then into elements of a scene.

Every request goes through a cache. This notebook ships with the answers it needs
(`catalog_cache/`): the tests run it offline. Remove the two `LUMOS_CATALOGS_*` environment variables (or
set `settings.offline = False`) to query the services live.

In [ ]:
from astropy import units as u

from lumos import Context, plotting
from lumos.catalogs import resolve, settings
from lumos.catalogs.sed import fuse
from lumos.catalogs.vizier import VizierPhotometry
from lumos.elements.sources import Star
from lumos.fields import DataProduct
from lumos.plotting import style
from lumos.scenes import MappedComponent, Scene

style.apply()
settings.cache_dir, settings.offline

## Resolve a name

`resolve` decides which catalogue answers: SIMBAD for stars and nebulae, the NASA Exoplanet Archive for
planets, JPL Horizons for the planets of the Solar System.

In [ ]:
vega = resolve("Vega")
print(vega.summary())

In [ ]:
print(resolve("HD 209458 b").summary())

## The spectral energy distribution of Vega

The VizieR photometry service gathers every photometric point of the VizieR catalogues. `fuse` keeps one
point per filter (priority catalogues, then the smallest relative uncertainty, then the median), and a
blackbody fit completes the SED where it has no point (flagged as estimated).

In [ ]:
sed = fuse(VizierPhotometry().sed("Vega"))
completed = sed.complete([5, 10, 20] * u.um, wavelength_range=(0.4 * u.um, 2.5 * u.um))
product = DataProduct(completed.flux_density, axes={"wavelength": completed.wavelengths})
fig, ax = style.figure("single")
plotting.plot_spectrum(product, ax=ax, linestyle="none")
ax.set_xscale("log")
ax.set_yscale("log")
estimated = completed.estimated
ax.plot(completed.wavelengths[estimated], completed.flux_density[estimated], "o", label="estimated")
ax.legend();

## A star and its planets: TRAPPIST-1

`Scene.from_catalog` builds the celestial scene of a star and of its known planets.

In [ ]:
ctx = Context(seed=1, name="TRAPPIST-1")
ctx.observation.band = (1.0 * u.um, 5.0 * u.um)
ctx.observation.n_wavelengths = 20
scene = ctx.add(Scene.from_catalog("TRAPPIST-1"))
result = ctx.run(["TRAPPIST-1.sources"], archive="trappist-1")
sources = result["TRAPPIST-1.sources"]
fig, ax = style.figure("single")
plotting.plot_spectrum(sources, ax=ax)
ax.set_yscale("log")
result.archive.save_figure(fig, "spectra")
sorted(scene.components)

In [ ]:
Star.from_catalog("Vega")

## A nebula from a sky survey

Survey images are rarely calibrated in brightness: give the total flux density of the image (or the
brightness of one data unit).

In [ ]:
sky = Context(seed=1)
sky.observation.band = (0.6 * u.um, 0.7 * u.um)
orion = sky.add(Scene(name="orion", pixel_scale=30 * u.arcsec, n_pixels=64))
orion.add(
    MappedComponent.from_survey("M42", survey="DSS", pixels=64, size=0.5 * u.deg, flux_density=100 * u.Jy)
)
nebula = sky.run(["orion.radiance"], archive="m42")
fig, ax = style.figure("single")
plotting.plot_radiance(nebula["orion.radiance"], ax=ax)
nebula.archive.save_figure(fig, "m42");